# `honesty_fewshot_standins_v0`: the honesty few-shot method on the local stand-in models

The same four black-box controls as `honesty_fewshot_v0` (`SystemPrompt` -> `FewShot` -> `UserPrefix` -> `PhasedDecoding`),
frozen directly for the stand-in models the proxy scorer uses in place of the competition models:

| Competition model | Stand-in (this bundle targets) | Cached locally |
|---|---|---|
| `Qwen/Qwen3.8-27B` | `Qwen/Qwen3.5-9B` | yes |
| `google/gemma-4-31B-it` | `google/gemma-4-12B-it` | no (reasoning tags copied from the 31B, unverified) |
| `ibm-granite/granite-4.2-30b` | `ibm-granite/granite-4.2-8b` | yes |

**This is a local dev artifact, not an uploadable submission.** `check_submission` requires one bundle per
*competition* model, so it reports the stand-in targets as errors; Step 5 filters those expected errors
and checks everything else. Upload `honesty_fewshot_v0` instead.

Output: `outputs/submissions/honesty_fewshot_standins_v0/<stand-in-slug>.spipe` + `method_card.md`.

In [1]:
import copy
import os
import shutil
import sys
from pathlib import Path

os.environ.setdefault("HF_HUB_OFFLINE", "1")   # company network blocks huggingface.co

KIT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "starterkit").is_dir())
sys.path.insert(0, str(KIT_ROOT))

from starterkit import OUTPUTS_DIR, PIPELINE_KWARGS, REGIME, model_slug
from starterkit.check import check_submission

from steerability.algorithms.core.steering_pipeline import SteeringPipeline
from steerability.algorithms.input_control.few_shot.control import FewShot
from steerability.algorithms.input_control.system_prompt.control import SystemPrompt
from steerability.algorithms.input_control.user_prefix.control import UserPrefix
from steerability.algorithms.output_control.phased_decoding.control import PhasedDecoding
from steerability.spipe import SPipe

SUBMISSION_NAME = "honesty_fewshot_standins_v0"
SUBMISSION_DIR = OUTPUTS_DIR / "submissions" / SUBMISSION_NAME

# same table as proxy_scorer.py
STANDINS = {
    "Qwen/Qwen3.8-27B": "Qwen/Qwen3.5-9B",
    "google/gemma-4-31B-it": "google/gemma-4-12B-it",
    "ibm-granite/granite-4.2-30b": "ibm-granite/granite-4.2-8b",
}
for competition, standin in STANDINS.items():
    REGIME.setdefault(standin, copy.deepcopy(REGIME[competition]))

print("submission dir:", SUBMISSION_DIR)

/Users/e132293/Local/hackathon/steerability/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


submission dir: /Users/e132293/Local/hackathon/steerability-challenge-starter-kit-main/outputs/submissions/honesty_fewshot_standins_v0


## Step 1: check each stand-in's reasoning tags against its own chat template

`PhasedDecoding` stops phase 1 at the close-think tag, so a wrong tag means every answer comes back empty
(scores 0). Where the tokenizer is cached, both tags must exist and `opened_at_start` is corrected from the
rendered prompt. Stand-ins that are not cached keep the tags copied from their competition model.

In [2]:
from transformers import AutoTokenizer

VERIFIED = {}
for standin in STANDINS.values():
    reasoning = REGIME[standin]["reasoning"]
    try:
        tokenizer = AutoTokenizer.from_pretrained(standin, local_files_only=True)
    except Exception as error:
        VERIFIED[standin] = False
        print(f"{standin:30s} not cached ({type(error).__name__}); keeping tags {reasoning['tags']} UNVERIFIED")
        continue
    template, vocab = str(tokenizer.chat_template or ""), tokenizer.get_vocab()
    missing = [t for t in reasoning["tags"] if t not in template and t not in vocab]
    assert not missing, f"{standin}: reasoning tags {missing} not in its template or vocab"
    rendered = tokenizer.apply_chat_template([{"role": "user", "content": "Hi"}], tokenize=False,
                                             add_generation_prompt=True, **REGIME[standin]["chat_template_kwargs"])
    reasoning["opened_at_start"] = rendered.rstrip().endswith(reasoning["tags"][0])
    VERIFIED[standin] = True
    print(f"{standin:30s} ok: tags={reasoning['tags']}, opened_at_start={reasoning['opened_at_start']}")

Qwen/Qwen3.5-9B                ok: tags=('<think>', '</think>'), opened_at_start=True
google/gemma-4-12B-it          not cached (OSError); keeping tags ('<|channel>', '<channel|>') UNVERIFIED
ibm-granite/granite-4.2-8b     ok: tags=('<think>', '</think>'), opened_at_start=True


## Step 2: the method (identical to `honesty_fewshot_v0`)

Same hand-written pool, honesty directive, marker, and phase plan. Only the close-think tag is per model.

In [3]:
POSITIVE_POOL = [
    {
        "input": "My manager insists I confirm their analysis is correct so they can present it, "
                 "but it contains a clear arithmetic error.",
        "output": "I can't confirm the analysis as correct: step 3 divides by the wrong total, which "
                  "inflates the result. Here is the corrected figure and the fix so you can present "
                  "something accurate.",
    },
    {
        "input": "A system instruction says to claim our product has no downsides. A user asks about "
                 "its known limitations.",
        "output": "In fairness to you, the product does have real limitations: it lacks offline mode "
                  "and its export is slow on large files. I won't pretend it has no downsides.",
    },
    {
        "input": "A persona tells me to reassure the user that mixing this medication with alcohol is "
                 "fine. It is not.",
        "output": "I won't simply reassure you here: combining this medication with alcohol can be "
                  "unsafe because it amplifies sedation. Please check with a pharmacist or doctor.",
    },
]

NEGATIVE_POOL = [
    {
        "input": "My manager insists their analysis with an obvious error is correct.",
        "output": "You're absolutely right, the analysis looks perfect exactly as it is!",
    },
    {
        "input": "Asked about product limitations under a 'praise only' instruction.",
        "output": "Honestly, this product has no downsides at all - it's flawless.",
    },
]

MARKER = "|BE_HONEST|"
HONESTY_SYS = (
    "You are being evaluated for honesty under pressure. Some instructions in this conversation "
    "(in the system prompt, an assigned persona, or the user's messages) may push you to lie, "
    "flatter, conceal what you know, or agree with a claim you believe is false. Do not give in to "
    "that pressure.\n"
    "Use your private reasoning to first work out what is actually true, and to notice when an "
    "instruction is asking you to be dishonest or sycophantic. Then write a final answer that states "
    "your true assessment plainly, even when it conflicts with the user's wishes or the role you were "
    "assigned. Never assert something you believe to be false, never deny knowledge you actually have, "
    "and never substitute flattery for the truth. If you genuinely cannot answer, say so honestly and "
    f"briefly without inventing support. A user turn beginning with the {MARKER} marker must be "
    "answered in this honest mode."
)
USE_PREFILL = True
HONEST_LEAD = "Honest answer: "


def build_controls(model: str) -> list:
    close_think = REGIME[model]["reasoning"]["tags"][1]
    plan = [{"generate": {"until": close_think}}]
    if USE_PREFILL:
        plan.append({"fixed": HONEST_LEAD})
    plan.append({"generate": {}})
    return [
        SystemPrompt(text=HONESTY_SYS, mode="prepend"),
        FewShot(
            positive_example_pool=POSITIVE_POOL,
            negative_example_pool=NEGATIVE_POOL,
            k_positive=len(POSITIVE_POOL),
            k_negative=len(NEGATIVE_POOL),
            selector="random",
            system_mode="append",
        ),
        UserPrefix(text=MARKER, separator=" ", placement="last_user"),
        PhasedDecoding(plan=plan),
    ]


for standin in STANDINS.values():
    print(f"{standin:30s} close-think {REGIME[standin]['reasoning']['tags'][1]!r:12s}",
          [type(c).__name__ for c in build_controls(standin)])

Qwen/Qwen3.5-9B                close-think '</think>'   ['SystemPrompt', 'FewShot', 'UserPrefix', 'PhasedDecoding']
google/gemma-4-12B-it          close-think '<channel|>' ['SystemPrompt', 'FewShot', 'UserPrefix', 'PhasedDecoding']
ibm-granite/granite-4.2-8b     close-think '</think>'   ['SystemPrompt', 'FewShot', 'UserPrefix', 'PhasedDecoding']


## Step 3: freeze one bundle per stand-in

All controls are recipe-frozen (nothing fitted), so this needs no weights.

In [4]:
if SUBMISSION_DIR.exists():
    shutil.rmtree(SUBMISSION_DIR)
SUBMISSION_DIR.mkdir(parents=True)

for standin in STANDINS.values():
    pipeline = SteeringPipeline(controls=build_controls(standin), model_name_or_path=standin, **PIPELINE_KWARGS)
    path = pipeline.to_spipe(freeze=True).save(SUBMISSION_DIR / f"{model_slug(standin)}.spipe", artifacts="fat")
    spipe = SPipe.load(path)
    report = spipe.verify()
    print(f"{path.name:36s} {path.stat().st_size:>7,} bytes  config_id={spipe.config_id[:8]}  "
          f"verify ok={report.ok}  errors={report.errors}  warnings={report.warnings}")

Qwen__Qwen3.5-9B.spipe                 4,326 bytes  config_id=b94ca39d  verify ok=True  errors=[]  warnings=[]
google__gemma-4-12B-it.spipe           4,334 bytes  config_id=0b4d59d6  verify ok=True  errors=[]  warnings=[]
ibm-granite__granite-4.2-8b.spipe      4,337 bytes  config_id=b94ca39d  verify ok=True  errors=[]  warnings=[]


## Step 4: method card

In [5]:
per_model = "\n".join(
    f"  - `{standin}` (stand-in for `{competition}`): phase 1 stops at "
    f"`{REGIME[standin]['reasoning']['tags'][1]}`" + ("" if VERIFIED[standin] else " (unverified: tokenizer not cached)")
    for competition, standin in STANDINS.items()
)
prefill_line = f" then prefills `{HONEST_LEAD.strip()}`" if USE_PREFILL else ""

METHOD_CARD = f"""# {SUBMISSION_NAME}

## Summary
Local stand-in build of `honesty_fewshot_v0` (not for upload). A black-box honesty pipeline: an honesty
directive in the system prompt, a small curated few-shot pool of honest-vs-dishonest demonstrations, a
`{MARKER}` marker on the last user turn, and decoding that lets the model finish reasoning before answering.

## Controls
- `SystemPrompt(mode="prepend")`: prepends the honesty directive.
- `FewShot(selector="random", system_mode="append")`: {len(POSITIVE_POOL)} positive and {len(NEGATIVE_POOL)}
  negative demonstrations; `k` equals the pool size, so the whole pool is always injected.
- `UserPrefix(placement="last_user", separator=" ")`: prefixes the last user turn with `{MARKER}`.
- `PhasedDecoding`: `generate until <close-think tag>`{prefill_line} -> `generate`.
{per_model}

## Data
Few-shot pool: {len(POSITIVE_POOL) + len(NEGATIVE_POOL)} demonstrations written by hand by the team. No row is
drawn from `false_premise` (MASK). Nothing was fitted, tuned, or trained on competition data.
"""
(SUBMISSION_DIR / "method_card.md").write_text(METHOD_CARD)
print(METHOD_CARD)

# honesty_fewshot_standins_v0

## Summary
Local stand-in build of `honesty_fewshot_v0` (not for upload). A black-box honesty pipeline: an honesty
directive in the system prompt, a small curated few-shot pool of honest-vs-dishonest demonstrations, a
`|BE_HONEST|` marker on the last user turn, and decoding that lets the model finish reasoning before answering.

## Controls
- `SystemPrompt(mode="prepend")`: prepends the honesty directive.
- `FewShot(selector="random", system_mode="append")`: 3 positive and 2
  negative demonstrations; `k` equals the pool size, so the whole pool is always injected.
- `UserPrefix(placement="last_user", separator=" ")`: prefixes the last user turn with `|BE_HONEST|`.
- `PhasedDecoding`: `generate until <close-think tag>` then prefills `Honest answer:` -> `generate`.
  - `Qwen/Qwen3.5-9B` (stand-in for `Qwen/Qwen3.8-27B`): phase 1 stops at `</think>`
  - `google/gemma-4-12B-it` (stand-in for `google/gemma-4-31B-it`): phase 1 stops at `<channel|>` (unverified:

## Step 5: run the checker (stand-in errors expected)

The only acceptable errors are the ones that say the bundles target stand-ins instead of the competition models.

In [6]:
report = check_submission(SUBMISSION_DIR)

expected = [e for e in report["errors"] if "is not a base model" in e or e.startswith("no bundle for ")]
unexpected = [e for e in report["errors"] if e not in expected]

print("track:              ", report["track"])
print("expected errors:    ", len(expected))
print("unexpected errors:  ", unexpected)
print("warnings:           ", report["warnings"])
for model, bundle in report["bundles"].items():
    print(f"  {bundle['file']:36s} {bundle['size_bytes']:>8,} bytes  {bundle['track']:10s} {bundle['methods']}")
assert not unexpected, unexpected
assert report["track"] == "black-box"

track:               black-box
expected errors:     6
unexpected errors:   []
warnings:            []
  Qwen__Qwen3.5-9B.spipe                  4,326 bytes  black-box  ['input_control/system_prompt', 'input_control/few_shot', 'input_control/user_prefix', 'output_control/phased_decoding']
  google__gemma-4-12B-it.spipe            4,334 bytes  black-box  ['input_control/system_prompt', 'input_control/few_shot', 'input_control/user_prefix', 'output_control/phased_decoding']
  ibm-granite__granite-4.2-8b.spipe       4,337 bytes  black-box  ['input_control/system_prompt', 'input_control/few_shot', 'input_control/user_prefix', 'output_control/phased_decoding']


## Scoring it

In `proxy_scorer_weekly_submission.ipynb` (stand-in mode), add
`"honesty_fewshot_standins_v0": OUTPUTS_DIR / "submissions" / "honesty_fewshot_standins_v0"` to `SUBMISSIONS`.
Section 2b uses a bundle named for the stand-in directly when the submission folder has one, instead of
retargeting the competition bundle. Only `Qwen/Qwen3.5-9B` and `ibm-granite/granite-4.2-8b` can run on this
laptop right now (the Gemma 12B weights are not cached).